In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
df = pd.read_csv("../data/indoorAir2.csv")

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s') 
df = df.sort_values(by='timestamp')

print(df.shape)
df.head()

In [ ]:
# # Hour cyclic features
# df["hour"] = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60

# df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
# df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# # Day of week cyclic features
# df["dayofweek"] = df["timestamp"].dt.dayofweek

# df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
# df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

# # Weekend feature
# df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# feature_df = df[
#     [
#         "ens160_aqi",
#         "ens160_tvoc",
#         "bme688_gas_resistance",
#         "bme688_pressure",
#         "scd41_temperature",
#         "scd41_humidity",
#         "timestamp",
#         "scd41_co2",
#         "hour_sin",
#         "hour_cos",
#         "dayofweek_sin",
#         "dayofweek_cos",
#         "is_weekend",
#         "station_id",
#     ]
# ].copy()


feature_df = df[
    [
        "ens160_aqi",
        "ens160_tvoc",
        "bme688_gas_resistance",
        "bme688_pressure",
        "scd41_temperature",
        "scd41_humidity",
        "timestamp",
        "scd41_co2",
        "station_id",
    ]
].copy()
print(feature_df.shape)
feature_df.head()

In [ ]:
print(feature_df.shape)
# Sort before merge_asof
feature_df = feature_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

# Create target lookup dataframe
target_df = feature_df[["station_id", "timestamp", "scd41_co2"]].copy()

target_df = target_df.rename(
    columns={
        "timestamp": "actual_target_timestamp",
        "scd41_co2": "target_co2_15min"
    }
)

# Required future timestamp
feature_df["target_timestamp"] = feature_df["timestamp"] + pd.Timedelta(minutes=15)

# Sort for merge_asof
feature_df = feature_df.sort_values(["target_timestamp", "station_id"]).reset_index(drop=True)
target_df = target_df.sort_values(["actual_target_timestamp", "station_id"]).reset_index(drop=True)

# Find nearest CO2 value around timestamp + 15 min
feature_df = pd.merge_asof(
    feature_df,
    target_df,
    left_on="target_timestamp",
    right_on="actual_target_timestamp",
    by="station_id",
    direction="nearest",
    tolerance=pd.Timedelta(minutes=2)
)

# Check actual difference from exact 15 min
feature_df["target_diff_seconds"] = (
    feature_df["actual_target_timestamp"] - feature_df["target_timestamp"]
).abs().dt.total_seconds()

# Remove rows without valid 15-min target
feature_df = feature_df.dropna(subset=["target_co2_15min"]).reset_index(drop=True)

feature_df = feature_df.drop(
    columns=[
        "target_timestamp",
        "actual_target_timestamp",
        "target_diff_seconds"
    ]
)

print(feature_df.shape)
feature_df.head()

In [ ]:
cols_to_fill = ["bme688_gas_resistance", "bme688_pressure"]

#feature_df = feature_df.sort_values("station_id").reset_index(drop=True)
feature_df = feature_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

feature_df[cols_to_fill] = (
    feature_df
    .groupby("station_id")[cols_to_fill]
    .transform(lambda x: x.interpolate(method="linear"))
)

feature_df[cols_to_fill] = (
    feature_df
    .groupby("station_id")[cols_to_fill]
    .ffill()
    .bfill()
)

print(feature_df.shape)
feature_df.head()

In [ ]:
numeric_cols = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_gas_resistance",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
    "scd41_co2",
    "target_co2_15min"
]

for col in numeric_cols:
    Q1 = feature_df[col].quantile(0.25)
    Q3 = feature_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    feature_df[col] = feature_df[col].clip(lower=lower, upper=upper)

print(feature_df.shape)
feature_df.head()

In [ ]:
outlier_summary = {}

for col in numeric_cols:
    Q1 = feature_df[col].quantile(0.25)
    Q3 = feature_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_count = ((feature_df[col] < lower) | (feature_df[col] > upper)).sum()

    outlier_summary[col] = {
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_percent": (outlier_count / len(feature_df)) * 100
    }

outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df)

In [ ]:
plot_cols = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
    "scd41_co2",
    "target_co2_15min"
]

feature_df[plot_cols].boxplot(figsize=(14, 6), rot=45)
plt.title("Boxplot Without Gas Resistance")
plt.show()

In [ ]:
train_stations = [1, 2, 4, 6]
val_stations = [3]
test_stations = [5]

train_df = feature_df[feature_df["station_id"].isin(train_stations)].copy()
val_df = feature_df[feature_df["station_id"].isin(val_stations)].copy()
test_df = feature_df[feature_df["station_id"].isin(test_stations)].copy()

total_rows = len(feature_df)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("Train percentage:", len(train_df) / total_rows * 100)
print("Val percentage:", len(val_df) / total_rows * 100)
print("Test percentage:", len(test_df) / total_rows * 100)

In [ ]:
target_col = "target_co2_15min"

"""feature_cols = [
    col for col in feature_df.columns
    if col != target_col
]"""
feature_cols = [
    col for col in feature_df.columns
    if col not in [target_col, "timestamp"]
]

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

train_df_scaled = train_df.copy()
val_df_scaled = val_df.copy()
test_df_scaled = test_df.copy()

train_df_scaled[feature_cols] = x_scaler.fit_transform(train_df[feature_cols])
val_df_scaled[feature_cols] = x_scaler.transform(val_df[feature_cols])
test_df_scaled[feature_cols] = x_scaler.transform(test_df[feature_cols])

train_df_scaled[[target_col]] = y_scaler.fit_transform(train_df[[target_col]])
val_df_scaled[[target_col]] = y_scaler.transform(val_df[[target_col]])
test_df_scaled[[target_col]] = y_scaler.transform(test_df[[target_col]])

print("Scaled X min/max:", train_df_scaled[feature_cols].min().min(), train_df_scaled[feature_cols].max().max())
print("Scaled y min/max:", train_df_scaled[target_col].min(), train_df_scaled[target_col].max())

In [ ]:
input_size = 60

def create_windows(data, feature_cols, target_col="target_co2_15min"):
    X_windows = []
    y_windows = []

    data = data.sort_values(["station_id", "timestamp"])

    for station_id, station_data in data.groupby("station_id"):
        X = station_data[feature_cols].values.astype("float32")
        y = station_data[target_col].values.astype("float32")

        for i in range(input_size, len(station_data)):
            X_windows.append(X[i - input_size:i])
            y_windows.append(y[i])

    return np.array(X_windows, dtype="float32"), np.array(y_windows, dtype="float32")


X_train, y_train = create_windows(train_df_scaled, feature_cols)
X_val, y_val = create_windows(val_df_scaled, feature_cols)
X_test, y_test = create_windows(test_df_scaled, feature_cols)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [ ]:
batch_size = 64

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
class NBeatsBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, theta_dim, num_layers=4):
        super(NBeatsBlock, self).__init__()

        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())

        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())

        self.fc = nn.Sequential(*layers)

        self.backcast_layer = nn.Linear(hidden_dim, input_dim)
        self.forecast_layer = nn.Linear(hidden_dim, theta_dim)

    def forward(self, x):
        h = self.fc(x)

        backcast = self.backcast_layer(h)
        forecast = self.forecast_layer(h)

        return backcast, forecast

In [ ]:
class NBeats(nn.Module):
    def __init__(self, input_size, num_features, hidden_dim=256, num_blocks=2, num_layers=3, horizon=1):
        super(NBeats, self).__init__()

        self.input_dim = input_size * num_features
        self.horizon = horizon

        self.blocks = nn.ModuleList([
            NBeatsBlock(
                input_dim=self.input_dim,
                hidden_dim=hidden_dim,
                theta_dim=horizon,
                num_layers=num_layers
            )
            for _ in range(num_blocks)
        ])

    def forward(self, x):
        x = x.reshape(x.size(0), -1)

        residual = x
        forecast = torch.zeros(
            x.size(0),
            self.horizon
        )

        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast
            forecast = forecast + block_forecast

        return forecast

In [ ]:
input_size = 60
num_features = X_train.shape[2]
horizon = 1

model = NBeats(
    input_size=input_size,
    num_features=num_features,
    hidden_dim=256,
    num_blocks=2,
    num_layers=3,
    horizon=horizon
)

print(model)

In [ ]:
criterion = nn.MSELoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)
# Add weight decay to reduce overfitting
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

In [ ]:
X_batch, y_batch = next(iter(train_loader))

y_pred = model(X_batch)

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("y_pred shape:", y_pred.shape)

print("y_pred min/max:", y_pred.min().item(), y_pred.max().item())

loss = criterion(y_pred, y_batch)
print("Loss:", loss.item())

In [ ]:
num_epochs = 30
patience = 3

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.6f} "
        f"Val Loss: {avg_val_loss:.6f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict()
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.eval()

test_predictions = []
test_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        y_pred = model(X_batch)

        test_predictions.append(y_pred.numpy())
        test_actuals.append(y_batch.numpy())

test_predictions = np.vstack(test_predictions)
test_actuals = np.vstack(test_actuals)

In [ ]:
test_predictions_original = y_scaler.inverse_transform(test_predictions)
test_actuals_original = y_scaler.inverse_transform(test_actuals)

In [ ]:
mae = mean_absolute_error(test_actuals_original, test_predictions_original)
mse = mean_squared_error(test_actuals_original, test_predictions_original)
rmse = np.sqrt(mse)
r2 = r2_score(test_actuals_original, test_predictions_original)

print("Test MAE:", mae)
print("Test MSE:", mse)
print("Test RMSE:", rmse)
print("Test R2:", r2)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_actuals_original, label="Actual CO2")
plt.plot(test_predictions_original, label="Predicted CO2")
plt.xlabel("Sample")
plt.ylabel("CO2")
plt.title("Actual vs Predicted CO2")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(test_actuals_original, test_predictions_original, alpha=0.3)
plt.xlabel("Actual CO2")
plt.ylabel("Predicted CO2")
plt.title("Actual vs Predicted CO2 Scatter")
plt.grid(True)
plt.show()